# 04 – Konjunkturklockan

**Syfte:** Göra om SCB:s konjunkturklocka från månadsdata 2000–2026 till en tabell med **en rad per år 2000–2025** och en kolumn per konjunkturindikator. Då kan tabellen kopplas till fertilitetsdata, som är årlig.

| | |
|---|---|
| **Källa** | SCB, Konjunkturklockan: 14 indikatorer, två mått, månadsvis |
| **Rådata** | `data/raw/konjunkturklockan.csv` (oförändrad, identisk med första versionen i git, commit `da9830b`) |
| **Resultat** | `data/processed/konjunktur_ar.csv` |
| **Beroenden** | Inga. Kan köras när som helst. |

Filen `CSV-files/konjunkturklockan.csv` i projektet är redan filtrerad för hand. Den här notebooken utgår från **originalutdraget** och gör hela filtreringen i kod. Då går stegen att följa och upprepa.

### Beslut som genomförs
1. **Behåll bara måttet *Konjunkturläge*.** *Förändring från föregående period* tas bort.
2. **Ta bort indikatorn *BNP kvartal*.** Den är en kvartalsserie som upprepats månadsvis och är redundant med *BNP-indikator månad*.
3. **Ta bort 2026**, som bara har 8 månader och ligger utanför fertilitetsdatans period.
4. **Slå ihop månader till år** som medelvärdet av årets 12 månader. Ofullständiga år blir `NaN`.

## 1. Inställningar

Sökvägar som i övriga notebooks, plus den period som ska ingå. Perioden följer fertilitetsdatan, 2000–2025.

In [1]:
from pathlib import Path

import pandas as pd

# Notebooken körs normalt från mappen Assignment.
# Körs den från projektroten letar vi upp Assignment ändå.
BAS = Path.cwd()
if not (BAS / "data" / "raw").is_dir() and (BAS / "Assignment").is_dir():
    BAS = BAS / "Assignment"

RAW = BAS / "data" / "raw"
PROCESSED = BAS / "data" / "processed"
PROCESSED.mkdir(parents=True, exist_ok=True)

assert RAW.is_dir(), "Hittar inte data/raw. Öppna notebooken från mappen Assignment."

FORSTA_AR = 2000
SISTA_AR = 2025

## 2. Läs in rådata

Filen är semikolonseparerad och kodad i UTF-8 med BOM. Varje rad är en indikator, ett mått och en månad. Månaden anges som text i formatet `2000M01`.

In [2]:
konj_ra = pd.read_csv(RAW / "konjunkturklockan.csv", sep=";", encoding="utf-8-sig")

display(konj_ra.head())
print("Form:", konj_ra.shape)
print(konj_ra.dtypes)

,indikator,tabellinnehåll,månad,Värde
0,BNP-indikator månad,Konjunkturläge,2000M01,-1.07
1,BNP-indikator månad,Konjunkturläge,2000M02,-0.84
2,BNP-indikator månad,Konjunkturläge,2000M03,-0.59
3,BNP-indikator månad,Konjunkturläge,2000M04,-0.29
4,BNP-indikator månad,Konjunkturläge,2000M05,0.01


Form: (8960, 4)
indikator             str
tabellinnehåll        str
månad                 str
Värde             float64
dtype: object


## 3. Kontrollera strukturen

Vi förväntar oss **14 indikatorer × 2 mått × 320 månader** (januari 2000–augusti 2026) **= 8 960 rader**, inga dubbletter och ett enhetligt månadsformat.

Rådatat har 134 saknade värden. Var de ligger och hur de hanteras visas i steg 7.

In [3]:
assert list(konj_ra.columns) == ["indikator", "tabellinnehåll", "månad", "Värde"]
assert konj_ra["indikator"].nunique() == 14
assert set(konj_ra["tabellinnehåll"]) == {"Konjunkturläge", "Förändring från föregående period"}
assert konj_ra["månad"].str.fullmatch(r"\d{4}M\d{2}").all()
assert not konj_ra.duplicated(["indikator", "tabellinnehåll", "månad"]).any()
assert len(konj_ra) == 14 * 2 * konj_ra["månad"].nunique()

print("Månader:", konj_ra["månad"].min(), "–", konj_ra["månad"].max(), f"({konj_ra['månad'].nunique()} st)")
print("Saknade värden:", konj_ra["Värde"].isna().sum())
print("\nIndikatorer:")
for namn in sorted(konj_ra["indikator"].unique()):
    print("  -", namn)

Månader: 2000M01 – 2026M08 (320 st)
Saknade värden: 134

Indikatorer:
  - Arbetade timmar
  - BNP kvartal
  - BNP-indikator månad
  - Detaljhandel sällanköpsvaror
  - Hushållens konfidensindikator
  - Hushållens konsumtion
  - Industrins orderingång
  - Nyregistrerade lastbilar
  - Nyregistrerade personbilar
  - Näringslivets efterfrågan
  - Näringslivets produktion
  - Sysselsättning
  - Varuexport
  - Varuimport


## 4. Behåll bara måttet Konjunkturläge

Konjunkturklockan redovisar två mått för varje indikator:

- **Konjunkturläge**: hur långt indikatorn ligger från sitt normala läge, på en standardiserad skala där 0 är normalläge. Måttet är alltså **redan avtrendat** och behöver ingen ytterligare avtrendning.
- **Förändring från föregående period**: hur läget har ändrats sedan förra månaden.

Projektet använder konjunkturläget, så förändringsmåttet tas bort. Kontrollen nedan visar att värdena ligger symmetriskt kring 0, med median nära 0 och ett spann på ungefär ±3. Det stämmer med en standardiserad skala.

In [4]:
konj = konj_ra.loc[konj_ra["tabellinnehåll"] == "Konjunkturläge"].drop(columns="tabellinnehåll")

print("Rader kvar:", len(konj))
print(f"Median: {konj['Värde'].median():.2f}   min: {konj['Värde'].min():.2f}   max: {konj['Värde'].max():.2f}")

Rader kvar: 4480
Median: -0.01   min: -3.49   max: 2.92


## 5. Ta bort BNP kvartal – redundant

*BNP kvartal* bygger på kvartalsvis nationalräkenskap men redovisas per månad. Kontrollen nedan visar två saker:

1. Inom varje kvartal är **alla tre månaderna exakt lika**. Serien har bara ett nytt värde per kvartal och är upprepad, inte månadsvis mätt.
2. Den följer **BNP-indikator månad** mycket nära (hög korrelation). BNP-indikatorn mäter samma sak men har ett nytt värde varje månad.

*BNP kvartal* tillför alltså ingen information och skulle bara ge dubblerade regler i ARM. Den tas bort.

In [5]:
bnp_kv = konj.loc[konj["indikator"] == "BNP kvartal"].sort_values("månad").reset_index(drop=True)
bnp_kv["kvartal"] = bnp_kv["månad"].str[:4] + "K" + ((bnp_kv["månad"].str[5:].astype(int) - 1) // 3 + 1).astype(str)
olika_per_kvartal = bnp_kv.groupby("kvartal")["Värde"].nunique()

bred = konj.pivot(index="månad", columns="indikator", values="Värde")
korrelation = bred["BNP kvartal"].corr(bred["BNP-indikator månad"])

print("Högsta antal olika värden inom ett kvartal:", olika_per_kvartal.max())
print(f"Korrelation med BNP-indikator månad: {korrelation:.3f}")
assert olika_per_kvartal.max() == 1

konj = konj.loc[konj["indikator"] != "BNP kvartal"]
print("Indikatorer kvar:", konj["indikator"].nunique())

Högsta antal olika värden inom ett kvartal: 1
Korrelation med BNP-indikator månad: 0.955
Indikatorer kvar: 13


## 6. Dela upp månaden och ta bort 2026

Texten `2000M01` delas upp i två heltal, `ar` och `manad`, så att det går att räkna med år.

Därefter tas **2026** bort av två skäl:

- Året har bara 8 månader (januari–augusti), och för juli–augusti saknas dessutom värden för flera indikatorer. Ett årsmedelvärde av ett halvt år går inte att jämföra med hela år.
- Fertilitetsdatan slutar 2025, så 2026 kan inte kopplas till något utfall.

In [6]:
konj = konj.assign(
    ar=konj["månad"].str[:4].astype(int),
    manad=konj["månad"].str[5:].astype(int),
)
assert konj["manad"].between(1, 12).all()

manader_2026 = sorted(konj.loc[konj["ar"] == 2026, "manad"].unique().tolist())
print("Månader som finns för 2026:", manader_2026)

konj = konj.loc[konj["ar"].between(FORSTA_AR, SISTA_AR)]
print("År kvar:", konj["ar"].min(), "–", konj["ar"].max())

Månader som finns för 2026: [1, 2, 3, 4, 5, 6, 7, 8]
År kvar: 2000 – 2025


## 7. Saknade månadsvärden

Tabellen visar vilka månader som saknar värden efter filtreringen. Alla hål ligger i **början av serien**. Tre indikatorer startar helt enkelt senare än år 2000:

- *Arbetade timmar* och *Sysselsättning* saknar hela 2000.
- *Näringslivets efterfrågan* saknar hela 2000 och januari–november 2001.

Det är alltså inget fel i data, utan serierna fanns inte. Hålen interpoleras inte, eftersom det inte finns något känt värde före dem.

In [7]:
saknas = konj.loc[konj["Värde"].isna()]
hal = saknas.groupby("indikator").agg(
    antal_manader=("månad", "size"),
    forsta=("månad", "min"),
    sista=("månad", "max"),
)
display(hal)

,antal_manader,forsta,sista
indikator,,,
Arbetade timmar,12,2000M01,2000M12
Näringslivets efterfrågan,23,2000M01,2001M11
Sysselsättning,12,2000M01,2000M12


## 8. Från månad till år

Fertilitet mäts per år, så konjunkturen måste också vara årlig. Årsvärdet blir **medelvärdet av årets 12 månader**.

Ett år räknas bara om **alla 12 månaderna finns**. Annars blir årsvärdet `NaN`. Ett medelvärde av några få månader skulle se lika säkert ut som ett helt år utan att vara det. *Näringslivets efterfrågan* 2001 bygger till exempel bara på december och skulle beskriva slutet av året i stället för hela året.

In [8]:
arsvarden = (
    konj.groupby(["indikator", "ar"])["Värde"]
    .agg(medel="mean", antal_manader="count")
    .reset_index()
)
arsvarden.loc[arsvarden["antal_manader"] < 12, "medel"] = float("nan")

ofullstandiga = arsvarden.loc[arsvarden["antal_manader"] < 12, ["indikator", "ar", "antal_manader"]]
print("Ofullständiga år som satts till NaN:")
display(ofullstandiga)

Ofullständiga år som satts till NaN:


,indikator,ar,antal_manader
0,Arbetade timmar,2000,0
208,Näringslivets efterfrågan,2000,0
209,Näringslivets efterfrågan,2001,1
260,Sysselsättning,2000,0


## 9. Bred form och korta kolumnnamn

Tabellen görs om till **en rad per år** med en kolumn per indikator. Indikatornamnen översätts till korta ASCII-namn via en uttrycklig tabell. Tabellen dokumenterar också vilket namn som hör till vilken indikator. Kontrollen ser till att ingen indikator faller bort eller saknar översättning.

In [9]:
KOLUMNNAMN = {
    "Arbetade timmar": "arbetade_timmar",
    "BNP-indikator månad": "bnp_indikator",
    "Detaljhandel sällanköpsvaror": "detaljhandel_sallankop",
    "Hushållens konfidensindikator": "hushallens_konfidens",
    "Hushållens konsumtion": "hushallens_konsumtion",
    "Industrins orderingång": "industrins_orderingang",
    "Nyregistrerade lastbilar": "nyreg_lastbilar",
    "Nyregistrerade personbilar": "nyreg_personbilar",
    "Näringslivets efterfrågan": "naringslivets_efterfragan",
    "Näringslivets produktion": "naringslivets_produktion",
    "Sysselsättning": "sysselsattning",
    "Varuexport": "varuexport",
    "Varuimport": "varuimport",
}
assert set(KOLUMNNAMN) == set(arsvarden["indikator"])

konj_ar = (
    arsvarden.pivot(index="ar", columns="indikator", values="medel")
    .rename(columns=KOLUMNNAMN)
    .rename_axis(columns=None)
    .reset_index()
    [["ar", *KOLUMNNAMN.values()]]
)
display(konj_ar.round(2).head())
print("Form:", konj_ar.shape)

,ar,arbetade_timmar,bnp_indikator,detaljhandel_sallankop,hushallens_konfidens,hushallens_konsumtion,industrins_orderingang,nyreg_lastbilar,nyreg_personbilar,naringslivets_efterfragan,naringslivets_produktion,sysselsattning,varuexport,varuimport
0,2000,NaN,0.08,0.52,1.79,0.45,0.51,0.52,0.95,NaN,0.20,NaN,0.45,0.98
1,2001,1.39,-0.14,-0.10,0.35,0.09,-0.62,-0.16,-0.76,NaN,0.03,0.80,-0.51,-0.16
2,2002,0.25,-0.42,-0.36,0.36,-0.24,-0.99,-0.50,-0.44,0.06,-0.47,0.51,-0.95,-0.98
3,2003,-1.03,-0.92,-0.61,0.01,-0.55,-0.84,-0.66,-0.00,-1.15,-0.63,-0.12,-0.79,-0.74
4,2004,-0.35,-0.26,-0.43,0.38,-0.93,-0.09,-0.31,0.13,-0.69,-0.30,-1.00,0.11,-0.43


Form: (26, 14)


## 10. Slutkontroll

Vi kontrollerar att varje år 2000–2025 finns exakt en gång och att det bara finns 4 saknade värden. Det är de år då serierna ännu inte startat (se steg 7–8).

**Att tänka på i analysen:** konjunkturen är **nationell**. Alla kommuner får samma värde ett givet år, så det finns bara 26 oberoende observationer av konjunkturen, inte en per kommun och år. Därför görs ingen fördröjning (*lag*) här. Hur många år konjunkturen ska ligga före fertiliteten är ett modellval som hör till analysen. Det görs när tabellerna kopplas ihop, till exempel `konj_ar.assign(ar=konj_ar["ar"] + 1)` för ett års fördröjning.

In [10]:
assert konj_ar["ar"].tolist() == list(range(FORSTA_AR, SISTA_AR + 1))

saknade = konj_ar.set_index("ar").isna().stack()
saknade = saknade[saknade].index.tolist()
print("Saknade årsvärden:", saknade)
assert len(saknade) == 4

Saknade årsvärden: [(2000, 'arbetade_timmar'), (2000, 'naringslivets_efterfragan'), (2000, 'sysselsattning'), (2001, 'naringslivets_efterfragan')]


## 11. Spara

In [11]:
utfil = PROCESSED / "konjunktur_ar.csv"
konj_ar.to_csv(utfil, index=False, encoding="utf-8-sig")

print(f"Sparade {len(konj_ar)} rader × {konj_ar.shape[1] - 1} indikatorer till {utfil.relative_to(BAS)}")

Sparade 26 rader × 13 indikatorer till data/processed/konjunktur_ar.csv
